## Loss Function: Measures how wrong the model's predictions are.

## Cross-Entropy: A loss function that measures how well the model predicts the correct next token.

## Perplexity: Measures how confused the model is when predicting text. Lower is better.


In [44]:
import torch
import torch.nn as nn


# GELU activation function banako class
class GELU(nn.Module):

    # class initialize garna use huncha
    def __init__(self):
        # parent nn.Module ko constructor call gareko
        super().__init__()

    # forward pass define gareko
    def forward(self, x):

        # GELU formula apply gareko
        return (
            0.5  # formula ko constant value
            * x  # input value multiply gareko
            * (
                1
                + torch.tanh(  # tanh activation use gareko
                    # sqrt(2/pi) calculate gareko
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    # x + 0.044715*x^3 calculate gareko
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )

In [45]:
import torch
import torch.nn as nn


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [46]:
from torch import nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [47]:
from torch import nn
import torch


class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        # d_out lai num_heads le divide garna milnu parxa
        # each head ko equal dimension ko lagi

        self.d_out = d_out

        self.num_heads = num_heads

        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # query projection layer banako

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # key projection layer banako

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # value projection layer banako

        self.out_proj = nn.Linear(d_out, d_out)
        # sabai heads combine pachi final projection garna use hunxa

        self.dropout = nn.Dropout(dropout)
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # causal mask banako
        # future tokens block garna use hunxa

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, token count, input dimension nikaleko

        keys = self.W_key(x)
        # input lai key vectors ma transform gareko

        queries = self.W_query(x)
        # input lai query vectors ma transform gareko

        values = self.W_value(x)
        # input lai value vectors ma transform gareko

        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # keys lai multiple heads ma split gareko

        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        # values lai multiple heads ma split gareko

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # queries lai multiple heads ma split gareko

        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)

        keys = keys.transpose(1, 2)
        # token ra head dimension swap gareko

        queries = queries.transpose(1, 2)
        # queries ko dimensions swap gareko

        values = values.transpose(1, 2)
        # values ko dimensions swap gareko

        attn_scores = queries @ keys.transpose(2, 3)
        # each head ko query ra key bich attention score nikaleko

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # required token size anusar mask select gareko

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # future token positions lai -inf banako
        # softmax pachi probability 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax attention apply gareko

        attn_weights = self.dropout(attn_weights)
        # attention weights ma dropout apply gareko

        context_vec = (attn_weights @ values).transpose(1, 2)
        # weighted sum garera context vector banako
        # transpose garera original order ma lyako

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # sabai heads combine garera reshape gareko

        context_vec = self.out_proj(context_vec)
        # final linear projection apply gareko

        return context_vec
        # final multi-head attention output return gareko

In [48]:
from torch import nn


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        # multi-head attention layer banako

        self.ff = FeedForward(cfg)
        # feed forward network banako

        self.norm1 = LayerNorm(cfg["emb_dim"])
        # attention block agadi layer normalization

        self.norm2 = LayerNorm(cfg["emb_dim"])
        # feed forward block agadi layer normalization

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        # shortcut connection ma dropout apply garna

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        # original input lai shortcut ma save gareko

        x = self.norm1(x)
        # attention block agadi normalization apply gareko

        x = self.att(x)
        # multi-head attention apply gareko
        # output shape [batch_size, num_tokens, emb_size]

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        # Shortcut connection for feed forward block
        shortcut = x
        # attention output lai shortcut ma save gareko

        x = self.norm2(x)
        # feed forward block agadi normalization apply gareko

        x = self.ff(x)
        # feed forward network apply gareko

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        return x
        # final transformer block output return gareko

In [49]:
from torch import nn
import torch


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # token embedding layer banako (word lai vector ma convert garna)

        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # positional embedding layer banako (sequence position encode garna)

        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # embeddings ma dropout apply garna (regularization ko lagi)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # multiple transformer blocks stack gareko (n_layers anusar)

        self.final_norm = LayerNorm(cfg["emb_dim"])
        # final layer normalization apply garna

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        # output linear layer (embedding → vocab logits)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        # input shape: batch size ra sequence length nikaleko

        tok_embeds = self.tok_emb(in_idx)
        # input tokens lai embedding vectors ma convert gareko

        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # sequence positions ko embedding banako

        x = tok_embeds + pos_embeds
        # token embedding + positional embedding combine gareko
        # shape: [batch_size, num_tokens, emb_size]

        x = self.drop_emb(x)
        # embeddings ma dropout apply gareko

        x = self.trf_blocks(x)
        # transformer blocks sequentially apply gareko

        x = self.final_norm(x)
        # final normalization apply gareko

        logits = self.out_head(x)
        # output linear layer apply garera vocab logits banako

        return logits
        # final output (logits for each token position) return gareko

In [50]:
import torch

GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 256,  # Shortened context length (orig: 1024)
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()
# Disable dropout during inference

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [51]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context

    ###Input batch:
    ###tensor([[6109, 3626, 6100,  345],
    ##[6109, 1110, 6622,  257]])

    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)  ### batch, n_tokens, vocab_size

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [52]:
import tiktoken


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())


start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## actual code ya bata ho yo vanda mathi jj model paila banako xa tei lako ho


In [ ]:
import torch

inputs = torch.tensor(
    [[16833, 3626, 6100], [40, 1107, 588]]  # ["every effort moves", "I really like"]
)
# inputs tensor banako → each row ek sentence ko token IDs

targets = torch.tensor(
    [
        [3626, 6100, 345],
        [1107, 588, 11311],
    ]  # ["effort moves you", "really like chocolate"]
)
# targets tensor banako → next-token prediction ko target IDs

In [ ]:
with torch.no_grad():
    logits = model(inputs)
    # inputs tensor lai model ma pathaera logits nikaleko
    # torch.no_grad() → gradient calculation disable gareko (inference mode)

probas = torch.softmax(logits, dim=-1)
# logits lai softmax apply garera probability distribution banako
# probability of each token in vocabulary

print(probas.shape)
# output shape print gareko → (batch_size, num_tokens, vocab_size)

torch.Size([2, 3, 50257])


In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
# probas tensor bata sabai bhanda highest probability ko token choose gareko
# argmax → max probability index nikaleko
# keepdim=True → dimension preserve gareko (shape consistent)

print("Token IDs:\n", token_ids)
# final predicted token IDs print gareko

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
# targets[0] → first batch ko target token IDs
# token_ids_to_text use garera IDs lai back to text convert gareko
# final target text print gareko

print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")
# token_ids[0] → first batch ko predicted token IDs
# flatten() → tensor lai 1D ma convert gareko
# token_ids_to_text use garera IDs lai text ma decode gareko
# final generated output text print gareko

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


## ya bata cross entropy loss find garney

## Cross-Entropy: A loss function that measures how well the model predicts the correct next token.


In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
# text_idx = 0 → first batch select gareko
# probas tensor bata first batch ko [0,1,2] positions ma
# targets[text_idx] → corresponding target token IDs select gareko
# target_probas_1 → model le target tokens lai assign gareko probabilities

print("Text 1:", target_probas_1)
# first text ko target token probabilities print gareko

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
# text_idx = 1 → second batch select gareko
# probas tensor bata second batch ko [0,1,2] positions ma
# targets[text_idx] → corresponding target token IDs select gareko
# target_probas_2 → model le target tokens lai assign gareko probabilities

print("Text 2:", target_probas_2)
# second text ko target token probabilities print gareko

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


In [ ]:
# Compute logarithm of all token probabilities
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
# target_probas_1 and target_probas_2 lai concatenate gareko
# torch.log → probabilities ko natural logarithm nikaleko
# log probabilities useful hunxa loss calculation (e.g. cross-entropy)

print(log_probas)
# final log probabilities print gareko

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [ ]:
# Calculate the average probability for each token
avg_log_probas = torch.mean(log_probas)
# log_probas tensor ko sabai values ko mean nikaleko
# average log probability → overall model confidence measure

print(avg_log_probas)
# final average log probability print gareko

tensor(-10.7940)


In [ ]:
neg_avg_log_probas = avg_log_probas * -1
# avg_log_probas lai negative banako
# negative average log probability → loss calculation ma use hunxa (e.g. NLL)

print(neg_avg_log_probas)
# final negative average log probability print gareko

tensor(10.7940)


In [ ]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)
# logits tensor ko shape print gareko
# 3D tensor → batch size, sequence length (num_tokens), vocabulary size

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)
# targets tensor ko shape print gareko
# 2D tensor → batch size, sequence length (expected next tokens)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [ ]:
logits_flat = logits.flatten(0, 1)
# logits tensor lai flatten gareko → first two dimensions (batch_size, num_tokens) merge
# result → 2D tensor (batch_size * num_tokens, vocab_size)

targets_flat = targets.flatten()
# targets tensor lai flatten gareko → 1D tensor (batch_size * num_tokens)
# result → ground-truth token IDs in flat form

print("Flattened logits:", logits_flat.shape)
# flattened logits ko shape print gareko → (batch_size * num_tokens, vocab_size)

print("Flattened targets:", targets_flat.shape)
# flattened targets ko shape print gareko → (batch_size * num_tokens)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
# logits_flat (predictions) vs targets_flat (ground-truth) compare gareko
# cross_entropy → negative log likelihood loss calculate garxa
# automatically softmax + log + average combine garera loss nikalxa

print(loss)
# final loss value print gareko → model performance measure

tensor(10.7940)


## ya bata perplexity find garney

## Perplexity: Measures how confused the model is when predicting text. Lower is better.


In [ ]:
perplexity = torch.exp(loss)
# loss (negative log likelihood) lai exponential gareko
# exp(loss) → perplexity measure nikaleko
# perplexity → model le average prediction ma kati uncertain xa bhanne metric

print(perplexity)
# final perplexity value print gareko

tensor(48725.8203)
